# Laboratorio 2: Minería Sujeta a Restricciones Logísticas en "Amazon Hub" 📦⚖️

**Contexto de Ingeniería:**
Trabajas en el centro de distribución de un gigante de comercio electrónico. El equipo de robótica quiere agrupar productos que suelen comprarse juntos dentro de los mismos contenedores ("Pods") para acelerar el empaque y ofrecer "Envío Express en una sola caja".

Sin embargo, el Gerente de Logística y Seguridad Industrial ha impuesto leyes físicas inquebrantables:
1. Las cajas de envío estándar se rompen si el paquete supera los **5.0 kg (5,000 gramos)**.
2. Por normas de protección civil, está estrictamente prohibido empacar **Sustancias Químicas/Inflamables** en la misma caja que **Alimentos** o **Electrónica con baterías de litio** (riesgo de contaminación o incendio).

Tu misión es extraer las reglas de asociación de los pedidos de este mes, sujetas a las restricciones físicas y de seguridad del almacén.

In [3]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. CATÁLOGO DE METADATOS LOGÍSTICOS: Peso en gramos y Categoría de Riesgo
catalogo_peso_g = {
    'Croquetas Perro 4kg': 4000,
    'Correa Mascota': 250,
    'Juguete Mordedera': 150,
    'Shampoo Antipulgas': 500,
    'PowerBank Litio': 400,
    'Cable Carga Rápida': 100,
    'Aerosol Limpiador': 600,
    'Paño Microfibra': 80,
    'Café en Grano 1kg': 1000,
    'Taza Térmica': 350,
    'Folleto Publicitario': 10
}

catalogo_riesgo = {
    'Croquetas Perro 4kg': 'Alimento',
    'Correa Mascota': 'General',
    'Juguete Mordedera': 'General',
    'Shampoo Antipulgas': 'Quimico',
    'PowerBank Litio': 'Litio',
    'Cable Carga Rápida': 'General',
    'Aerosol Limpiador': 'Inflamable',
    'Paño Microfibra': 'General',
    'Café en Grano 1kg': 'Alimento',
    'Taza Térmica': 'General',
    'Folleto Publicitario': 'Basura_Operativa'
}

# 2. HISTORIAL DE PEDIDOS DEL ALMACÉN
pedidos_almacen = [
    ['Croquetas Perro 4kg', 'Correa Mascota', 'Juguete Mordedera', 'Folleto Publicitario'],
    ['Croquetas Perro 4kg', 'Shampoo Antipulgas', 'Correa Mascota'],
    ['PowerBank Litio', 'Cable Carga Rápida', 'Aerosol Limpiador', 'Paño Microfibra'],
    ['Café en Grano 1kg', 'Taza Térmica', 'Folleto Publicitario'],
    ['Croquetas Perro 4kg', 'Café en Grano 1kg', 'Taza Térmica'],
    ['PowerBank Litio', 'Cable Carga Rápida', 'Paño Microfibra', 'Folleto Publicitario'],
    ['Aerosol Limpiador', 'Paño Microfibra', 'Shampoo Antipulgas'],
    ['Croquetas Perro 4kg', 'Correa Mascota', 'Juguete Mordedera'],
    ['Café en Grano 1kg', 'Taza Térmica', 'Aerosol Limpiador'],
    ['PowerBank Litio', 'Cable Carga Rápida', 'Aerosol Limpiador']
]

### Reto 1: Poda de Ruido y Extracción Base

1. El artículo `'Folleto Publicitario'` se mete gratis en casi todas las cajas, ensuciando las métricas. Elimínalo de tu matriz booleana **antes** de ejecutar `apriori()`.
2. Ejecuta `apriori()` con un `min_support = 0.2` y genera las reglas con una `confidence` mínima de `0.6`.

In [4]:
te = TransactionEncoder()
df_almacen = pd.DataFrame(te.fit_transform(pedidos_almacen), columns=te.columns_)
df_almacen_limpio = df_almacen.drop(columns=['Folleto Publicitario'])

itemsets_logistica = apriori(df_almacen_limpio, min_support=0.2, use_colnames=True)
reglas_logistica = association_rules(itemsets_logistica, metric="confidence", min_threshold=0.6)

print(f"Total de reglas candidatas: {len(reglas_logistica)}")
display(reglas_logistica[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())

Total de reglas candidatas: 32


,antecedents,consequents,support,confidence,lift
0,(Cable Carga Rápida),(Aerosol Limpiador),0.2,0.666667,1.666667
1,(Paño Microfibra),(Aerosol Limpiador),0.2,0.666667,1.666667
2,(PowerBank Litio),(Aerosol Limpiador),0.2,0.666667,1.666667
3,(Paño Microfibra),(Cable Carga Rápida),0.2,0.666667,2.222222
4,(Cable Carga Rápida),(Paño Microfibra),0.2,0.666667,2.222222


### Reto 2: Mapeo de Restricciones Físicas (Peso y Categorías de Riesgo)

Para evaluar cada regla, necesitamos saber cuánto pesa el combo completo (`antecedents` | `consequents`) y qué etiquetas de riesgo se están mezclando dentro de la misma caja.

**Tu tarea:**
1. Crea la columna `peso_total_g` sumando el peso de todos los artículos involucrados en la regla.
2. Crea la columna `categorias_en_caja` que guarde un conjunto (`set`) con los tipos de riesgo involucrados en la regla (usando `catalogo_riesgo`).

In [5]:
def calcular_peso_total(row):
    items_totales = row['antecedents'] | row['consequents']
    return sum(catalogo_peso_g[item] for item in items_totales)

def obtener_riesgos(row):
    items_totales = row['antecedents'] | row['consequents']
    return set(catalogo_riesgo[item] for item in items_totales)

reglas_logistica['peso_total_g'] = reglas_logistica.apply(calcular_peso_total, axis=1)
reglas_logistica['categorias_en_caja'] = reglas_logistica.apply(obtener_riesgos, axis=1)

display(reglas_logistica[['antecedents', 'consequents', 'peso_total_g', 'categorias_en_caja']].head(8))

,antecedents,consequents,peso_total_g,categorias_en_caja
0,(Cable Carga Rápida),(Aerosol Limpiador),700,"{General, Inflamable}"
1,(Paño Microfibra),(Aerosol Limpiador),680,"{General, Inflamable}"
2,(PowerBank Litio),(Aerosol Limpiador),1000,"{Inflamable, Litio}"
3,(Paño Microfibra),(Cable Carga Rápida),180,{General}
4,(Cable Carga Rápida),(Paño Microfibra),180,{General}
5,(Cable Carga Rápida),(PowerBank Litio),500,"{General, Litio}"
6,(PowerBank Litio),(Cable Carga Rápida),500,"{General, Litio}"
7,(Café en Grano 1kg),(Taza Térmica),1350,"{General, Alimento}"


### Reto 3: El Filtro de Seguridad Industrial (Exclusión Mutua y Peso Máximo)

Filtra el DataFrame `reglas_logistica` para obtener las `reglas_aprobadas` que cumplan **simultáneamente** con las 3 leyes del almacén:

1. **Restricción Anti-Monótona de Peso:** El `peso_total_g` debe ser **menor o igual a 4,500 gramos** (dejamos 500g de margen para el cartón y relleno).
2. **Restricción de Contaminación Cruzada:** Prohibido que en `categorias_en_caja` coexistan `'Alimento'` y `'Quimico'` (o `'Inflamable'`) al mismo tiempo.
3. **Restricción de Incendio:** Prohibido que en `categorias_en_caja` coexistan `'Litio'` e `'Inflamable'` al mismo tiempo.

In [6]:
def es_caja_segura(conjunto_riesgos):
    if 'Alimento' in conjunto_riesgos and ('Quimico' in conjunto_riesgos or 'Inflamable' in conjunto_riesgos):
        return False
    if 'Litio' in conjunto_riesgos and 'Inflamable' in conjunto_riesgos:
        return False
    return True

reglas_aprobadas = reglas_logistica[
    (reglas_logistica['peso_total_g'] <= 4500) &
    (reglas_logistica['categorias_en_caja'].apply(es_caja_segura)) &
    (reglas_logistica['lift'] > 1.0)
]

print(f"Reglas seguras y aprobadas para empaque conjunto: {len(reglas_aprobadas)}")
display(reglas_aprobadas[['antecedents', 'consequents', 'confidence', 'lift', 'peso_total_g', 'categorias_en_caja']])

Reglas seguras y aprobadas para empaque conjunto: 26


,antecedents,consequents,confidence,lift,peso_total_g,categorias_en_caja
0,(Cable Carga Rápida),(Aerosol Limpiador),0.666667,1.666667,700,"{General, Inflamable}"
1,(Paño Microfibra),(Aerosol Limpiador),0.666667,1.666667,680,"{General, Inflamable}"
3,(Paño Microfibra),(Cable Carga Rápida),0.666667,2.222222,180,{General}
4,(Cable Carga Rápida),(Paño Microfibra),0.666667,2.222222,180,{General}
5,(Cable Carga Rápida),(PowerBank Litio),1.000000,3.333333,500,"{General, Litio}"
6,(PowerBank Litio),(Cable Carga Rápida),1.000000,3.333333,500,"{General, Litio}"
7,(Café en Grano 1kg),(Taza Térmica),1.000000,3.333333,1350,"{General, Alimento}"
8,(Taza Térmica),(Café en Grano 1kg),1.000000,3.333333,1350,"{General, Alimento}"
9,(Croquetas Perro 4kg),(Correa Mascota),0.750000,2.500000,4250,"{General, Alimento}"
10,(Correa Mascota),(Croquetas Perro 4kg),1.000000,2.500000,4250,"{General, Alimento}"


### 🧠 Reto 4: Defensa de Ingeniería Algorítmica (Responde aquí)

**1. Propiedad Anti-Monótona (Optimización de C++ / Python):**
La restricción `peso_total_g <= 4500` es matemáticamente **Anti-Monótona**. 
Imagina que el algoritmo está evaluando el par `('Croquetas Perro 4kg', 'Café en Grano 1kg')`, el cual ya pesa `5,000g`. 
Explica por qué la propiedad anti-monótona le permite al algoritmo abortar inmediatamente y **no evaluar** el trío `('Croquetas Perro 4kg', 'Café en Grano 1kg', 'Taza Térmica')`, ahorrando millones de operaciones de CPU.
> **Tu Respuesta:** El peso solo crece al agregar artículos, si el par ya rompe el límite, todo superconjunto también lo rompe, así que el algoritmo poda la rama sin evaluarla.

**2. Auditoría de Descarte:**
Menciona al menos **dos combos específicos** que tenían un excelente *Lift* y *Confianza* en los datos de los clientes, pero que tu filtro bloqueó exitosamente. Explica qué tragedia operativa o accidente evitaste en cada uno.
> **Tu Respuesta:** PowerBank Litio con Aerosol Limpiador, lift 1.67 Y Aerosol Limpiador más Cable Carga Rápida con PowerBank Litio, lift 3.33, en ambos se evitó meter una batería de litio junto a un inflamable, osea, prevenimos un posible incencio.